# AutoEval: Vector Search Quality Evaluation

This notebook evaluates vector search quality by:
1. Generating synthetic queries from your document corpus
2. Running vector searches with those queries
3. Using an LLM judge to score relevance of results
4. Analyzing the results to understand search quality


---

## Quick Start

**Time estimate:** ~20-30 minutes total

**What you'll need:**
- A Vector Search endpoint name
- A Vector Search index name
- The source table for your index

**Steps:**
1. **Edit cells 2-3** - Fill in your required config values
2. **Edit cell 7** - Add a few example queries (optional but recommended)
3. **Run All** - Everything else runs automatically

---

## Install Dependencies

This cell installs the autoeval library. If you encounter issues:

**Troubleshooting:**
1. Download the wheel file from the provided location
2. Upload it to your Databricks workspace (e.g., `/Workspace/Users/your.email@company.com/`)
3. Update the path below to point to your uploaded wheel file

**Expected output:** Installation completes with no errors, then Python restarts.

In [ ]:
# Upload the autoeval_lib wheel to your workspace and update this path
WHEEL_PATH = ""

%pip install --force-reinstall databricks-vectorsearch tqdm $WHEEL_PATH --quiet
dbutils.library.restartPython()

## Required Configuration

**You MUST update these values before running.** These are specific to your Vector Search setup.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# REQUIRED: You MUST update these values
# ══════════════════════════════════════════════════════════════════════════════

# Your Vector Search index name (format: catalog.schema.index_name)
# Find at: Catalog > [catalog] > [schema] > [index]
# NOTE: Do NOT include backticks here, even if your schema has special characters
INDEX_NAME = ""

# Columns containing text content for query generation and search.
# These should be the column(s) in your source table that contain searchable text.
QUERY_COLUMNS = [""]

# Model used for embedding documents/queries.
# Required for self-managed indexes, leave empty to auto-derive from managed index
EMBEDDING_MODEL = ""

# Cache table for storing query embeddings and LLM scores (format: catalog.schema.table)
# This speeds up repeated evaluation runs and reduces LLM API costs
QUERY_CACHE_TABLE = ""

## Optional Configuration

**These defaults work well for most cases.** Only change if you have specific requirements.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# OPTIONAL: These defaults work for most cases
# ══════════════════════════════════════════════════════════════════════════════

# LLM endpoint for synthetic query generation
# Default: databricks-gemini-3-flash (fast, cost-efficient, good reasoning)
QUERY_GENERATION_LLM_ENDPOINT = "databricks-gemini-3-flash"

# LLM endpoint for relevance scoring (the "LLM judge")
# Default: databricks-gemini-3-flash (fast, cost-efficient, good reasoning)
RELEVANCE_JUDGE_LLM_ENDPOINT = "databricks-gemini-3-flash"

# How do your users typically search?
#   - "keyword": Short queries like "helicopter rotor types"
#   - "natural": Full questions like "What are the types of helicopter rotors?"
#   - "mixed": Combination of both styles
QUERY_STYLE = "mixed"

# Number of queries to generate for evaluation
EVAL_QUERYSET_SIZE = 200

# Number of results to fetch per query
NUM_RESULTS = 10

# Query types to evaluate: FULL_TEXT, ANN, HYBRID (case insensitive)
QUERY_TYPES = ["FULL_TEXT", "ANN", "HYBRID"]

# Additional columns to retrieve beyond primary_key + QUERY_COLUMNS
ADDITIONAL_RETRIEVAL_COLUMNS = []

# Reranker configuration
ENABLE_RERANKER = True
ADDITIONAL_RERANKER_COLUMNS = []

# Seed for reproducible document sampling
RANDOM_SEED = 42

# Number of documents to sample for generating example document/query pairs
NUM_DOCS_TO_SAMPLE = 20

# MLFlow logging (logs params, metrics, artifacts, and traces to MLflow)
ENABLE_MLFLOW_LOGGING = True

# MLFlow experiment path (leave empty to auto-generate based on index name)
MLFLOW_EXPERIMENT_PATH = ""

## Validate Configuration

This cell validates your configuration and auto-derives settings from the index metadata.

In [ ]:
from autoeval_lib import AutoEvalConfig

# Create and validate configuration
config = AutoEvalConfig(
    index_name=INDEX_NAME,
    query_columns=QUERY_COLUMNS,
    query_generation_llm_endpoint=QUERY_GENERATION_LLM_ENDPOINT,
    relevance_judge_llm_endpoint=RELEVANCE_JUDGE_LLM_ENDPOINT,
    embedding_model=EMBEDDING_MODEL,
    query_style=QUERY_STYLE,
    num_samples=NUM_DOCS_TO_SAMPLE,
    num_queries=EVAL_QUERYSET_SIZE,
    num_results=NUM_RESULTS,
    query_types=QUERY_TYPES,
    additional_retrieval_columns=ADDITIONAL_RETRIEVAL_COLUMNS,
    enable_reranker=ENABLE_RERANKER,
    additional_reranker_columns=ADDITIONAL_RERANKER_COLUMNS,
    query_cache_table=QUERY_CACHE_TABLE,
    random_seed=RANDOM_SEED,
    enable_mlflow_logging=ENABLE_MLFLOW_LOGGING,
    mlflow_experiment_path=MLFLOW_EXPERIMENT_PATH,
)

# Validate and auto-derive fields from index
config.validate()

## Sample Documents & Generate Example Queries

This cell samples documents from your corpus and generates example queries.
Review these to understand what kinds of queries the model creates.

In [ ]:
from autoeval_lib import QueryGenerator

# Initialize QueryGenerator with config
generator = QueryGenerator(config=config)

# Generate example queries (both natural and keyword style)
samples_df = generator.sample_and_generate_examples()

# Display the samples with generated queries
display(samples_df)

## Provide Few-Shot Examples

Based on the examples above, provide your own document-query pairs.
These guide the model to generate better queries for your use case.

**Tips:**
- Copy 3-5 document snippets from the table above
- Write the queries your users would actually search for
- Use triple quotes (`"""`) for multi-line documents - this avoids syntax errors
- (Optional) Change the query style to match your query examples

In [ ]:
# Provide your few-shot examples
# Copy document snippets from above and write the queries you'd want for them
FEW_SHOT_EXAMPLES = [
    {
        "document": "All Along The Watchtower by Jimi Hendrix Songfacts. 1  This was written and originally recorded by Bob Dylan in 1967, but it was the Jimi Hendrix cover that made the song famous. Many other artists have covered it, including Eric Clapton, Neil Young, U2, Dave Matthews Band and The Grateful Dead.",
        "query": "who wrote all along the watchtower"
    },
    {
        "document": "by Tomo Â· December 12, 2010. Basketball was invented in 1891 by the Canadian physical instructor James Naismith. He was born in Almonte, Ontario in Canada on November 6, 1861. At the tender age of nine he lost both his parents within three weeks.He got educated at Mcgill University and Presbyterian College at Montreal.asketball was invented in 1891 by the Canadian physical instructor James Naismith. He was born in Almonte, Ontario in Canada on November 6, 1861.",
        "query": "basketball origin"
    },
    {
        "document": "The DASH Eating Plan. 1  The DASH eating plan is rich in fruits, vegetables, fat-free or low-fat milk and milk products, whole. grains, fish, poultry, beans, seeds, and nuts. It also. contains less sodium; sweets, added sugars, and. beverages containing sugar; fats; and red meats. than the typical American diet. This heart-healthy.",
        "query": "dash diet"
    },
    # Add more examples based on the samples you saw above
    # {
    #     "document": "...",
    #     "query": "..."
    # },
]

# Update the generator with your examples
generator.set_few_shot_examples(FEW_SHOT_EXAMPLES)
# Optional: Change query style at runtime if needed
# generator.set_query_style("keyword")

## Generate Full Query Set

Now we'll generate queries at scale using your few-shot examples.

In [ ]:
# Generate queries using config defaults
queries_df = generator.generate_queries()

# Preview generated queries
display(queries_df.limit(20))

## Run Evaluation

This runs vector search for each query using specified query types and scores the relevance of results using the LLM.

**This typically takes 15-25 minutes for 200 queries.** The cell will show progress as it runs.

In [ ]:
from autoeval_lib import MultiQueryEvaluator

# Initialize evaluator with config
evaluator = MultiQueryEvaluator(config=config)

# Run evaluation - traces and metrics logged automatically to MLflow
results = evaluator.evaluate(queries_df)

## Ranking Metrics

### How to Interpret These Results

**Relevance Scores (0-3 scale):**
- **3 = Highly Relevant:** Exactly what the user wanted
- **2 = Relevant:** Useful but not perfect
- **1 = Marginally Relevant:** Somewhat related
- **0 = Not Relevant:** Not useful for the query

**Which query type wins?**
- **ANN winning:** Your documents have strong semantic content; users benefit from meaning-based search
- **FULL_TEXT winning:** Your users search with specific terms; keyword matching is effective
- **HYBRID winning:** Your use case benefits from both semantic and keyword matching

**If scores are very close (within confidence intervals):**
The differences may not be statistically meaningful. Consider:
- Running with more queries (`EVAL_QUERYSET_SIZE`)
- Looking at specific categories of queries where differences emerge

### Metric descriptions.
The following metrics evaluate search quality from different perspectives:

- **Recall@k**: Fraction of queries where the original document appears in top k results, for example Recall@10 measures how often the original document appears in the top 10 results. 
  [Learn more](https://en.wikipedia.org/wiki/Precision_and_recall)

- **Precision@k**: Fraction of top k results that are relevant (score >= 2).
  [Learn more](https://en.wikipedia.org/wiki/Precision_and_recall)

- **MRR (Mean Reciprocal Rank)**: Average of 1/rank of the first relevant result.
  Higher MRR means relevant results appear earlier.
  [Learn more](https://en.wikipedia.org/wiki/Mean_reciprocal_rank)

- **NDCG@k (Normalized Discounted Cumulative Gain)**: Measures ranking quality by comparing actual result ordering to ideal ordering. Uses graded relevance (0-3) and position discounting. NDCG=1.0 means results are perfectly ranked (highest scores first).
  [Learn more](https://en.wikipedia.org/wiki/Discounted_cumulative_gain)

- **DCG@k (Discounted Cumulative Gain)**: Measures total relevance utility using graded scores (0-3). Unlike NDCG which normalizes by the ideal ranking, DCG reflects absolute retrieval quality. Higher values mean more relevant content was retrieved. See theoretical maximum values below for interpretation.
  [Learn more](https://en.wikipedia.org/wiki/Discounted_cumulative_gain)

- **MAP@k (Mean Average Precision)**: Measures ranking quality by computing average precision at each relevant document position, then averaging across all queries. Rewards systems that rank relevant documents higher.
  [Learn more](https://en.wikipedia.org/wiki/Evaluation_measures_(information_retrieval)#Mean_average_precision)

### DCG Interpretation Guide

DCG (Discounted Cumulative Gain) measures total relevance utility using graded scores (0-3).
Higher is better - there is no upper bound.

**Theoretical DCG_max@k (assuming max relevance for all retrieved documents):**

Formula: `DCG_max@k = 3 × Σ(1/log2(r+1))` for r=1..k

| k   | DCG_max |
|-----|---------|
| 1   | 3.00    |
| 3   | 6.39    |
| 5   | 8.85    |
| 10  | 13.63   |

All metrics include 95% confidence intervals (±).

## Comparison Summary

Side-by-side comparison of metrics across all query types.

In [ ]:
print("=" * 60)
print("METRICS COMPARISON ACROSS QUERY TYPES")
print("=" * 60)

# Side-by-side metrics table
print("\nKey Metrics:")
display(evaluator.compare_metrics())

# DCG@k comparison
print("\nDCG@k Comparison (Discounted Cumulative Gain):")
display(evaluator.compare_dcg())

# NDCG@k comparison
print("\nNDCG@k Comparison (using graded relevance 0-3):")
display(evaluator.compare_ndcg())

# MAP@k comparison
print("\nMAP@k Comparison (Mean Average Precision):")
display(evaluator.compare_map())

# Precision@k comparison
print("\nPrecision@k Comparison (relevant = score >= 2):")
display(evaluator.compare_precision())

# Recall@k comparison
print("\nRecall@k Comparison:")
display(evaluator.compare_recall())

## Metric Visualizations

Interactive plots showing metrics with 95% confidence intervals across query types.

In [ ]:
# Plot all metrics with confidence intervals
# Plots will be skipped if plotly is not installed

for name, fig in evaluator.plot_all_metrics():
    if fig:
        print("=" * 25 + f" {name} " + "=" * 25)
        display(fig)

## Detailed Analysis per Query Type

Drill down into metrics and examples for each query type.

In [ ]:
for query_type, result in results.items():
    print(f"\n{'='*60}")
    print(f"DETAILED ANALYSIS: {query_type}")
    print(f"{'='*60}")

    # Summary metrics
    print(result.analyzer.summary())

    # Score distribution
    print("\nScore Distribution:")
    display(result.analyzer.score_distribution())

## Top & Bottom Performing Queries

View the best and worst performing queries for each query type.

In [ ]:
for query_type, result in results.items():
    print(f"\n{'='*60}")
    print(f"{query_type}: Top 5 Queries (by avg relevance)")
    print(f"{'='*60}")
    display(result.analyzer.top_queries(5))

    print(f"\n{query_type}: Bottom 5 Queries (by avg relevance)")
    display(result.analyzer.bottom_queries(5))

## High & Low Relevance Examples

View examples of highly relevant and low relevance results for each query type.

In [ ]:
for query_type, result in results.items():
    print(f"\n{'='*60}")
    print(f"{query_type}: High Relevance Examples (score=3)")
    print(f"{'='*60}")
    display(result.analyzer.high_relevance_examples(5))

    print(f"\n{query_type}: Low Relevance Examples (score<=1)")
    display(result.analyzer.low_relevance_examples(5))

## MLflow Results

When `enable_mlflow_logging=True`, evaluation results are automatically logged to MLflow during `evaluate()`.
Each query type gets its own MLflow run containing:
- **Traces**: One trace per query with search results and LLM scores
- **Metrics**: avg_relevance_score, recall@k, highly_relevant_pct, etc.
- **Artifacts**: score_distribution.json, top_queries.json, bottom_queries.json

In [ ]:
# View MLflow experiment path
if config.enable_mlflow_logging and evaluator.experiment_path:
    print(f"MLflow Experiment: {evaluator.experiment_path}")
    print(f"\nTo view results:")
    print(f"  1. Go to MLflow Experiments in Databricks")
    print(f"  2. Search for: {evaluator.experiment_path}")
    print(f"  3. Each query type (FULL_TEXT, ANN, HYBRID) has its own run")